In [1]:
import pandas as pd

In [2]:
def read_tbl_leo():
    df = pd.read_csv("../data/input/ca-2026-post-raw.csv")
    df = df.drop(columns=["officer_id", "term_code", "rank", "app_status"])
    df = df.rename(
        columns={
            "employment_start_date": "start_date",
            "employment_end_date": "end_date",
            "term_code_desc": "separation_reason",
            "POST_ID": "person_nbr",
            "agency": "agency_name",
        }
    )
    return df


df = read_tbl_leo()

df

,person_nbr,officer_name,agency_name,start_date,end_date,separation_reason
0,B03-J69,"ANDERSEN, MARK B",SAN DIEGO CO SD,12/1/1995,3/27/2020,Retired
1,A06-W53,"SIRACUSA, ANTHONY W",RIVERSIDE PD,4/18/2001,9/24/2001,Other
2,A06-W53,"SIRACUSA, ANTHONY W",RIVERSIDE PD,9/24/2001,10/19/2022,Retired
3,C91-P47,"NGUYEN, KELLY",SAN JOSE PD,6/26/2022,2/5/2023,Status Change
4,C91-P47,"NGUYEN, KELLY",SAN JOSE PD,2/5/2023,4/30/2023,Resigned
...,...,...,...,...,...,...
470783,A15-J11,"KIMBLE, SCOT E",PERRIS PD,4/17/1995,4/11/1996,Other
470784,A15-J11,"KIMBLE, SCOT E",BANNING PD,11/5/2001,12/31/2009,Resigned
470785,A15-J11,"KIMBLE, SCOT E",MCFARLAND PD,11/30/2011,4/8/2019,Resigned
470786,A15-J11,"KIMBLE, SCOT E",ARVIN PD,4/10/2019,3/13/2020,Resigned


In [ ]:
def clean_sep_reason(df):
    df.loc[:, "separation_reason"] = (
        df.separation_reason.fillna("")
        .str.lower()
        .str.replace(r"unknown", "", regex=False)
    )
    return df


# Multi-word phrases — applied BEFORE single tokens so longer matches win.
MULTI_WORD = [
    (r"\bunif(ied)? sch(oo)?l? dist(rict)?\b", "unified school district"),
    (r"\bunified school dist\b", "unified school district"),
    (r"\bcomm college\b", "community college"),
    (r"\bcomm(unity)? coll(ege)? dist\b", "community college district"),
    (r"\bcomm ctr\b", "communications center"),
    (r"\bcomm dept\b", "communications department"),
    (r"\b(emerg|emer) comm\b", "emergency communications"),
    (r"\bemer mgt\b", "emergency management"),
    (r"\bdept of\b", "department of"),
    (r"\bint'l\b", "international"),
    (r"\bjr college\b", "junior college"),
    (r"\bhigh sch(ool)? dist\b", "high school district"),
    (r"\bpark rang\b", "park rangers"),
    (r"\bden bd\b", "dental board"),
    (r"\bwelfare fd\b", "welfare fraud"),
    (r"\bmed(ical)? exam\b", "medical examiner"),
    (r"\ble support\b", "law enforcement support"),
    (r"\bsan fran\b", "san francisco"),
    (r"\bunion hs\b", "union high school"),
]

# Single-token abbreviations — applied AFTER multi-word.
TOKENS = {
    r"\bcsu\b": "california state university",
    r"\buc\b": "university of california",
    r"\bccd\b": "community college district",
    r"\bcc\b": "community college",
    r"\busd\b": "unified school district",
    r"\buhsd\b": "union high school district",
    r"\bdps\b": "department of public safety",
    r"\bdept\b": "department",
    r"\bdpt\b": "department",
    r"\bdist\b": "district",
    r"\bunif\b": "unified",
    r"\bschl?\b": "school",
    r"\bco\b": "county",
    r"\bso\b": "sheriff's office",
    r"\bsd\b": "sheriff's department",
    r"\bda\b": "district attorney",
    r"\bdoj\b": "department of justice",
    r"\bofc\b": "office",
    r"\boes\b": "office of emergency services",
    r"\binvest\b": "investigations",
    r"\binv\b": "investigations",
    r"\benf\b": "enforcement",
    r"\bpub\b": "public",
    r"\bsfty\b": "safety",
    r"\bsvcs?\b": "services",
    r"\bsvs\b": "services",
    r"\baff\b": "affairs",
    r"\bmed\b": "medical",
    r"\bmngd\b": "managed",
    r"\bcntr?l\b": "control",
    r"\blegis\b": "legislature",
    r"\bsgt\b": "sergeant",
    r"\bbch\b": "beach",
    r"\bhbr\b": "harbor",
    r"\brwy\b": "railway",
    r"\brr\b": "railroad",
    r"\bmar\b": "marshal",
    r"\bmrshl\b": "marshal",
    r"\bcor\b": "coroner",
    r"\bauth\b": "authority",
    r"\breg\b": "regional",
    r"\bctr\b": "center",
    r"\bairpt\b": "airport",
    r"\btrns\b": "transit",
    r"\btran\b": "transit",
    r"\bnorthrn\b": "northern",
    r"\bmuncpl\b": "municipal",
    r"\bdev\b": "developmental",
    r"\butil\b": "utility",
    r"\bagcy\b": "agency",
    r"\bmgt\b": "management",
    r"\bemer\b": "emergency",
    r"\bcomm\b": "communications",
    r"\bhs\b": "high school",
    r"\bhss\b": "health and human services",
    r"\bgen\b": "general",
    r"\basst\b": "assistance",
    r"\bsrv\b": "services",
    r"\bhum\b": "human",
    r"\bfrd\b": "fraud",
    r"\bwfraud\b": "welfare fraud",
    r"\bfin\b": "financial",
    r"\bprot\b": "protection",
    r"\binnov\b": "innovation",
    r"\byth\b": "youth",
    r"\btob\b": "tobacco",
    r"\bops\b": "operations",
    r"\brec\b": "recreation",
    r"\bchil\b": "child support",
    r"\bser\b": "services",
    r"\ble\b": "law enforcement",
    r"\bcslb\b": "contractors state license board",
    r"\bmantec\b": "manteca",
    r"\bstock\b": "stockton",
    r"\brvrside\b": "riverside",
    r"\bsm\b": "santa maria",
}

# Genuine irregulars — full-string replacements.
IRREGULAR = {
    "cal fire": "california department of forestry and fire protection",
    "cal fire - office of state fire marshal": "california department of forestry and fire protection - office of state fire marshal",
    "cal - oes": "california office of emergency services",
    "supreme court of ca": "supreme court of california",
    "lawrence berkeley lab": "lawrence berkeley national laboratory",
    "sacramento city unified": "sacramento city unified school district",
    "san diego harbor pd, port of": "port of san diego harbor police department",
    "alameda belt line railroad police": "alameda belt line railroad police department",
    "other-in-state": "",
    "other-out-of-state": "",
    "unknown/non-affiliate": "",
}


def clean_agency_name(df):
    s = df["agency_name"].fillna("").astype(str).str.lower()

    # 1. strip trailing legacy C-codes (e.g. "biggs pd  c-04")
    s = s.str.replace(r"\s+c-\d{2}\s*$", "", regex=True)

    # 2. handle full-string irregulars first
    s = s.replace(IRREGULAR)

    # 3. multi-word phrases
    for pat, repl in MULTI_WORD:
        s = s.str.replace(pat, repl, regex=True)

    # 4. single-token abbreviations
    for pat, repl in TOKENS.items():
        s = s.str.replace(pat, repl, regex=True)

    # 5. structural suffixes — anchored, run last
    s = s.str.replace(
        r" county sheriff's (department|office)(/coroner)?$",
        " county sheriff's office",
        regex=True,
    )
    s = s.str.replace(r"\bpd\b", "police department", regex=True)
    s = s.str.replace(r"^ca\b", "california", regex=True)

    # comma-separated suffixes -> " - " (sub-unit notation)
    s = s.str.replace(r"\s*,\s*", " - ", regex=True)

    # LA county safety hsd/isd have no comma in source — insert dash
    s = s.str.replace(r"\bsafety (hsd|isd)\b", r"safety - \1", regex=True)

    # collapse double spaces
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()

    df.loc[:, "agency_name"] = s
    return df

In [4]:
def clean_dates(df):
    for c in ("start_date", "end_date"):
        df[c] = (
            pd.to_datetime(df[c], errors="coerce")
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    return df

In [5]:
SUFFIX_RE = r"(?i)\s+(jr|sr|ii|iii|iv|v)\.?\s*$"


def clean_officer_name(df):
    df = df[
        df["officer_name"].str.strip().str.lower() != "name withheld"
    ].copy()
    parts = df["officer_name"].str.split(",", n=1, expand=True)
    df["last_name"] = parts[0].str.strip().str.title()
    rest = parts[1].fillna("").str.strip()
    df["suffix"] = (
        rest.str.extract(SUFFIX_RE, expand=False).fillna("").str.upper()
    )
    rest = rest.str.replace(SUFFIX_RE, "", regex=True)
    df[["first_name", "middle_name"]] = (
        rest.str.split(n=1, expand=True)
        .fillna("")
        .apply(lambda c: c.str.title())
    )
    return df.drop(columns=["officer_name"])

In [ ]:
def read_corrections():
    df = pd.read_csv(
        "../data/input/ca-2023-clean-corrections.csv", dtype={"person_nbr": str}
    )
    df = df.rename(
        columns={"agcy_name": "agency_name", "middle_initial": "middle_name"}
    )
    for col in ("first_name", "middle_name", "last_name"):
        df[col] = df[col].fillna("").astype(str).str.title()
    df["suffix"] = ""
    df["separation_reason"] = ""
    return df[
        [
            "person_nbr",
            "first_name",
            "middle_name",
            "last_name",
            "suffix",
            "agency_name",
            "start_date",
            "end_date",
            "separation_reason",
            "type",
        ]
    ]

In [ ]:
df = (
    df.pipe(clean_sep_reason)
    .pipe(clean_agency_name)
    .pipe(clean_dates)
    .pipe(clean_officer_name)
)
df["type"] = "POLICE"

df = pd.concat([df, read_corrections()], ignore_index=True, sort=False)

for col in [
    "start_date",
    "end_date",
    "middle_name",
    "suffix",
    "separation_reason",
]:
    df[col] = (
        df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .replace({"NaT": "", "None": "", "nan": ""})
    )
df = df[df["start_date"] != ""]
df = df.drop_duplicates(subset=["person_nbr", "agency_name", "start_date"])

df

,person_nbr,agency_name,start_date,end_date,separation_reason,last_name,suffix,first_name,middle_name,type
0,B03-J69,san diego county sheriff's office,1995-12-01,2020-03-27,retired,Andersen,,Mark,B,POLICE
1,A06-W53,riverside police department,2001-04-18,2001-09-24,other,Siracusa,,Anthony,W,POLICE
2,A06-W53,riverside police department,2001-09-24,2022-10-19,retired,Siracusa,,Anthony,W,POLICE
3,C91-P47,san jose police department,2022-06-26,2023-02-05,status change,Nguyen,,Kelly,,POLICE
4,C91-P47,san jose police department,2023-02-05,2023-04-30,resigned,Nguyen,,Kelly,,POLICE
...,...,...,...,...,...,...,...,...,...,...
608436,293927,934: HIGH DESERT STATE PRISON,2019-04-01,,,Herrera Ortega,,Lucio,,CORRECTIONS
608437,293928,048: RA MCGEE CORR TRAIN CNTR -AKA- RICHARD A ...,2023-03-20,,,Diaz Penaloza,,Jose,F,CORRECTIONS
608438,293928,095: SAN QUENTIN STATE PRISON,2023-06-19,,,Diaz Penaloza,,Jose,F,CORRECTIONS
608439,293929,048: RA MCGEE CORR TRAIN CNTR -AKA- RICHARD A ...,2019-10-15,,,Gamboa,,Cristian,G,CORRECTIONS


In [ ]:
df.to_csv("../output/ca_index.csv", index=False)
print(f"Wrote {len(df):,} rows to ../output/ca_index.csv")